# Notebook 1 : Chargement des données, découpage journalier, agrégation, split panel/cible, stats descriptives

In [1]:
%reload_ext autoreload
%autoreload 2
import sys, pathlib
ROOT = pathlib.Path.cwd().parent                  
if str(ROOT) not in sys.path:                     
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider, SelectionSlider, Dropdown

import online_dp as dp
from online_dp.config import Config
from online_dp.cache import compute_or_load
from online_dp import data, metrics, basis, gam, mechanisms, vfast, viz

cfg = Config(
    data_dir=str(ROOT.parent / "data" / "DataDiffusionDeepCourboGen") + "/",   
    cache_dir=str(ROOT / "cache"),
    n_panel=500, n_target=50, seed=0,                
)
dp.cache.CACHE = pathlib.Path(cfg.cache_dir)       # cache partagé par tous les notebooks
cfg

Config(data_dir='/home/G70186/data/DataDiffusionDeepCourboGen/', cache_dir='/home/G70186/onlinedp_stage26/cache', N=1000, n_panel=500, n_target=50, seed=0, calendar_start='2022-10-02 20:00:00', slots_per_day=48, pmax=47, n_groups=100, fit_subsample=10000, nmf_max_iter=500, nmf_tol=0.0001, clip_quantile=0.95, delta_dp=1e-05)

## Construire/recharger le dataset complet
Tout le notebook 1 tient dans `data.build_dataset`, mis en cache.

In [2]:
D = compute_or_load(cfg.key('dataset'), lambda: data.build_dataset(cfg))
df_label, df_temp = D['df_label'], D['df_temp']
df_daily, df_agg  = D['df_daily'], D['df_agg']
panel_users, target_users = D['panel_users'], D['target_users']
panel_profiles, panel_tensor, panel_ids = D['panel_profiles'], D['panel_tensor'], D['panel_ids']
print(f"df_daily {df_daily.shape} | df_agg {df_agg.shape} | panel {len(panel_users)} | cible {len(target_users)}")

[cache] 'dataset__N1000_np500_nt50_s0' rechargé (joblib).
df_daily (362000, 48) | df_agg (362, 48) | panel 500 | cible 50


## Statistiques descriptives

In [3]:
# Répartition Power × ToU des N foyers séléectionnés 
sampled_ids = df_daily.index.get_level_values('household_id').unique()
counts = data.label_distribution(df_label.loc[sampled_ids])      
viz.plot_power_tou_3d(counts, total=len(sampled_ids)).show()
print(counts.rename(index=lambda p: f"{p} kVA",
                    columns={0: 'Base', 1: 'Heures Creuses', 2: 'Tempo'}))

ToU     Base  Heures Creuses  Tempo
Power                              
6 kVA     92             198     35
9 kVA    135             273     35
12 kVA    57             147     28


## Profils individuels, agrégats


In [4]:
slots = np.arange(48)
SEED  = 0

prof = df_daily.sample(5, random_state=SEED)
days = df_agg.sample(5, random_state=SEED + 1).sort_index()

n_hh = df_daily.index.get_level_values('household_id').nunique()
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f"5 profils journaliers individuels (parmi {n_hh} foyers)",
                    f"Agregat L(t) - 5 jours (cible de {len(target_users)} foyers)"))

for i, ((hh, d), row) in enumerate(prof.iterrows()):
    fig.add_trace(go.Scatter(x=slots, y=row.values, mode='lines', legend='legend',
                  name=f"{hh} - {pd.to_datetime(d):%Y-%m-%d}",
                  line=dict(color=viz.PALETTE_PROFILS[i], width=1.8)), row=1, col=1)

for i, (d, row) in enumerate(days.iterrows()):
    fig.add_trace(go.Scatter(x=slots, y=row.values, mode='lines', legend='legend2',
                  name=f"{pd.to_datetime(d):%Y-%m-%d}",
                  line=dict(color=viz.PALETTE_AGG[i], width=2.4)), row=1, col=2)

fig.update_xaxes(domain=[0.00, 0.34], title_text="creneau demi-horaire (0-47)", row=1, col=1)
fig.update_xaxes(domain=[0.55, 0.86], title_text="creneau demi-horaire (0-47)", row=1, col=2)
fig.update_yaxes(rangemode='tozero', title_text="consommation (kVA)", row=1, col=1)
fig.update_yaxes(rangemode='tozero', title_text="agregat L(t) (kVA)",  row=1, col=2)
fig.layout.annotations[0].update(x=0.17,  xanchor='center')
fig.layout.annotations[1].update(x=0.705, xanchor='center')
fig.update_layout(
    height=460, width=1300, template='plotly_white',
    title=dict(text="Profils individuels vs agregat cible L(t)", x=0.5,
               font=dict(size=15, color=viz.COLOR_REAL)),
    legend =dict(title='foyer - jour', x=0.36, xanchor='left', y=1, yanchor='top',
                 bgcolor='rgba(255,255,255,0.9)', bordercolor='#d5dbdb', borderwidth=1),
    legend2=dict(title='jour',         x=0.88, xanchor='left', y=1, yanchor='top',
                 bgcolor='rgba(255,255,255,0.9)', bordercolor='#d5dbdb', borderwidth=1),
    margin=dict(t=70, b=60, l=70, r=30))
fig.show()

## Heatmap de la conso sur l'agrégat : joux $\times$ créneau 

In [5]:
Z     = df_agg.values                 # (T, 48)
dates = df_agg.index                  # DatetimeIndex
slots = np.arange(48)
hhmm  = [f"{h//2:02d}:{'30' if h % 2 else '00'}" for h in slots]

fig = go.Figure(go.Heatmap(
    z=Z, x=slots, y=dates,
    colorscale='Viridis',
    colorbar=dict(title='kVA', thickness=14, len=0.85),
    customdata=np.tile(hhmm, (len(dates), 1)),
    hovertemplate='%{y|%d %b %Y} - %{customdata}<br>%{z:.2f} kVA<extra></extra>'))

fig.update_xaxes(title_text='heure de la journee',
                 tickvals=[0, 12, 24, 36, 47],
                 ticktext=['00:00', '06:00', '12:00', '18:00', '23:30'])
fig.update_yaxes(title_text='', autorange='reversed', tickformat='%b', dtick='M1')
fig.update_layout(
    title=dict(text="Agregat L(t) : charge par jour et par creneau", x=0.5,
               font=dict(size=15, color='#1d3557')),
    height=640, width=560, template='plotly_white',
    margin=dict(t=60, b=50, l=55, r=20))
fig.show()

## Thermosensibilité 

In [6]:
sampled_ids = df_daily.index.get_level_values('household_id').unique()
load_daily = df_daily.groupby(level='date').mean().mean(axis=1)
load_daily.index = pd.to_datetime(load_daily.index)
cols       = df_temp.columns.intersection(sampled_ids)
temp_30    = df_temp[cols].mean(axis=1)
temp_daily = temp_30.groupby(temp_30.index.normalize()).mean()
temp_daily.index = pd.to_datetime(temp_daily.index)
df_ts = pd.DataFrame({'temp': temp_daily, 'load': load_daily}).dropna().sort_values('temp')
bins  = np.arange(np.floor(df_ts.temp.min()), np.ceil(df_ts.temp.max()) + 2, 2.0)
trend = (df_ts.assign(b=pd.cut(df_ts.temp, bins))
              .groupby('b', observed=True)
              .agg(t=('temp', 'mean'), l=('load', 'mean')).dropna())
month = df_ts.index.month
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_ts.temp, y=df_ts.load, mode='markers', name='jour',
    marker=dict(size=6, color=month, colorscale=viz.SCALE_SAISON, cmin=1, cmax=12,
                opacity=0.82, line=dict(width=0.4, color='white'),
                colorbar=dict(title='mois', thickness=14, len=0.8,
                              tickvals=[1, 4, 7, 10, 12],
                              ticktext=['jan', 'avr', 'jui', 'oct', 'dec'])),
    text=[d.strftime('%d %b') for d in df_ts.index],
    hovertemplate='%{text}<br>%{x:.1f} degC - %{y:.3f} kVA<extra></extra>'))
fig.add_trace(go.Scatter(
    x=trend.t, y=trend.l, mode='lines+markers', name='moyenne par 2 degC',
    line=dict(color=viz.COLOR_REAL, width=2.5), marker=dict(size=5)))
fig.update_layout(
    title=dict(text="Thermosensibilite : conso journaliere vs temperature (N=1000 foyers)",
               x=0.5, font=dict(size=15, color=viz.COLOR_REAL)),
    xaxis_title='temperature moyenne journaliere (°C)',
    yaxis_title='charge moyenne journaliere (kVA)',
    height=480, width=720, template='plotly_white',
    legend=dict(x=0.98, y=0.98, xanchor='right', yanchor='top',
                bgcolor='rgba(255,255,255,0.8)', bordercolor='#d5dbdb', borderwidth=1),
    margin=dict(t=70, b=55, l=65, r=20))
fig.show()